# 02 - BEACON contrastive training

Align AlphaEarth with POI semantics and visitation behavior at POI locations, then retain the AlphaEarth projector for image-only deployment.

**Projectors**

- `AEProjector`, shared by local and context views: a 64 -> 256 -> 128 main branch, a gated 64 -> 128 residual branch, and a 128 -> 256 -> 128 refinement MLP
- `TextProjector`: 384 -> 128
- `MobilityProjector`: 184 -> 256 -> 128

**Objective:** weighted sum of symmetric CLIP-style InfoNCE losses with temperature 0.07.

| Alignment pair | Weight |
|---|---:|
| AlphaEarth local - text | 0.35 |
| AlphaEarth local - mobility | 0.35 |
| Text - mobility | 0.15 |
| AlphaEarth local - context | 0.15 |

**Inputs:** the training table from notebook 01 and the 64-band Houston AlphaEarth tiles.

**Outputs:** the BEACON checkpoint and 128-band regional BEACON embedding tiles used by notebook 03.


## 0. Configuration


In [ ]:
from pathlib import Path
import os
import copy
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

PROJECT_DIR = Path(os.environ.get("BEACON_PROJECT_DIR", Path.cwd())).expanduser().resolve()
CONTRASTIVE_DIR = PROJECT_DIR / "contrastive_learning"

TRAIN_TABLE = CONTRASTIVE_DIR / "houston_poi_ae_text_mobility_cov1_train.parquet"
CKPT_PATH = CONTRASTIVE_DIR / "beacon_model.pt"
AE64_DIR = PROJECT_DIR / "Houston_AE_2024"                                   # Input: 64-band AlphaEarth rasters
BEACON_DIR = CONTRASTIVE_DIR / "houston_beacon_embeddings_128"       # Output: 128-band BEACON rasters

SEED = 42
SPLIT = (0.8, 0.1, 0.1)
BATCH_SIZE = 512
NUM_EPOCHS = 100
LR = 1e-3
WEIGHT_DECAY = 1e-4
TEMPERATURE = 0.07
LOSS_WEIGHTS = {"ae_text": 0.35, "ae_mobility": 0.35, "text_mobility": 0.15, "ae_ae": 0.15}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

## 1. Data loading and preprocessing

- L2-normalize each local and context AlphaEarth vector.
- Standardize the four mobility scalar features with training-set statistics. Keep the 168 hour-of-week and 12 monthly distributions unchanged.
- Use the text embeddings without additional preprocessing.
- Split POIs randomly into 80% training, 10% validation, and 10% test sets.


In [ ]:
df = pd.read_parquet(TRAIN_TABLE)

AE_SMALL_COLS = sorted(c for c in df.columns if c.startswith("small_A"))
AE_CONTEXT_COLS = sorted(c for c in df.columns if c.startswith("context_A"))
TEXT_COLS = sorted(c for c in df.columns if c.startswith("text_emb_"))
MOB_SCALAR_COLS = ["visit_total_log", "entropy_168", "weekday_weekend_gap", "monthly_cv"]
MOB_COLS = (
    MOB_SCALAR_COLS
    + sorted(c for c in df.columns if c.startswith("weekly_168_norm_"))
    + sorted(c for c in df.columns if c.startswith("monthly_12_norm_"))
)
assert (len(AE_SMALL_COLS), len(AE_CONTEXT_COLS), len(TEXT_COLS), len(MOB_COLS)) == (64, 64, 384, 184)

idx = np.random.default_rng(SEED).permutation(len(df))
n_train, n_val = int(len(df) * SPLIT[0]), int(len(df) * SPLIT[1])
splits = {
    "train": df.iloc[idx[:n_train]].reset_index(drop=True),
    "val": df.iloc[idx[n_train:n_train + n_val]].reset_index(drop=True),
    "test": df.iloc[idx[n_train + n_val:]].reset_index(drop=True),
}

mob_mean = splits["train"][MOB_SCALAR_COLS].mean().to_numpy(np.float32)
mob_std = splits["train"][MOB_SCALAR_COLS].std(ddof=0).replace(0, 1.0).to_numpy(np.float32)


def l2_normalize_np(X, eps=1e-6):
    X = X.astype(np.float32, copy=False)
    return X / np.clip(np.linalg.norm(X, axis=1, keepdims=True), eps, None)


class POIDataset(Dataset):
    def __init__(self, frame):
        mob = np.array(frame[MOB_COLS], dtype=np.float32)
        mob[:, :4] = (mob[:, :4] - mob_mean) / mob_std
        self.arrays = {
            "ae_small": l2_normalize_np(frame[AE_SMALL_COLS].to_numpy()),
            "ae_context": l2_normalize_np(frame[AE_CONTEXT_COLS].to_numpy()),
            "text": frame[TEXT_COLS].to_numpy(np.float32),
            "mobility": mob,
        }

    def __len__(self):
        return len(self.arrays["ae_small"])

    def __getitem__(self, i):
        return {k: torch.from_numpy(v[i]) for k, v in self.arrays.items()}


loaders = {
    name: DataLoader(POIDataset(frame), batch_size=BATCH_SIZE,
                     shuffle=(name == "train"), drop_last=(name == "train"))
    for name, frame in splits.items()
}
print({k: len(v) for k, v in splits.items()})

## 2. Model


In [ ]:
def l2_normalize(x):
    return F.normalize(x, p=2, dim=-1)


class AEProjector(nn.Module):
    """Project 64 to 128 dimensions with a main branch, gated residual, L2 normalization, and refinement MLP."""

    def __init__(self, input_dim=64, hidden_dim=256, output_dim=128, p_drop=0.0):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(input_dim, 256), nn.ReLU(inplace=True), nn.Linear(256, output_dim))
        self.shortcut = nn.Linear(input_dim, output_dim)
        self.gate = nn.Parameter(torch.tensor([-4.0]))   # sigmoid(-4) is approximately 0.018, so the residual starts weak
        self.post_mlp = nn.Sequential(
            nn.Linear(output_dim, hidden_dim), nn.GELU(), nn.Dropout(p_drop), nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        z = l2_normalize(self.proj(x) + torch.sigmoid(self.gate) * self.shortcut(x))
        return l2_normalize(self.post_mlp(z))


class TextProjector(nn.Module):
    def __init__(self, input_dim=384, output_dim=128):
        super().__init__()
        self.proj = nn.Linear(input_dim, output_dim)

    def forward(self, x):
        return l2_normalize(self.proj(x))


class MobilityProjector(nn.Module):
    def __init__(self, input_dim=184, hidden_dim=256, output_dim=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, output_dim))

    def forward(self, x):
        return l2_normalize(self.net(x))


class BEACON(nn.Module):
    def __init__(self):
        super().__init__()
        self.ae_projector = AEProjector()
        self.text_projector = TextProjector()
        self.mobility_projector = MobilityProjector()

    def forward(self, ae_small, ae_context, text, mobility):
        return {
            "ae_small": self.ae_projector(ae_small),
            "ae_context": self.ae_projector(ae_context),
            "text": self.text_projector(text),
            "mobility": self.mobility_projector(mobility),
        }


def clip_loss(z1, z2, temperature=TEMPERATURE):
    logits = z1 @ z2.T / temperature
    labels = torch.arange(z1.size(0), device=z1.device)
    return 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels))


def compute_losses(z):
    losses = {
        "ae_text": clip_loss(z["ae_small"], z["text"]),
        "ae_mobility": clip_loss(z["ae_small"], z["mobility"]),
        "text_mobility": clip_loss(z["text"], z["mobility"]),
        "ae_ae": clip_loss(z["ae_small"], z["ae_context"]),
    }
    losses["total"] = sum(LOSS_WEIGHTS[k] * v for k, v in losses.items())
    return losses

## 3. Training

Train with AdamW, cosine learning-rate annealing, and gradient clipping at 1.0. Retain the checkpoint with the lowest total validation loss and report final metrics on the test set.


In [ ]:
def to_device(batch):
    return {k: v.to(DEVICE) for k, v in batch.items()}


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    sums, n = {}, 0
    for batch in loader:
        losses = compute_losses(model(**to_device(batch)))
        for k, v in losses.items():
            sums[k] = sums.get(k, 0.0) + v.item()
        n += 1
    return {k: v / n for k, v in sums.items()}


set_seed(SEED)
model = BEACON().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

history = []
best_val, best_state = float("inf"), None
t0 = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_sums, n = {}, 0
    for batch in loaders["train"]:
        losses = compute_losses(model(**to_device(batch)))
        optimizer.zero_grad()
        losses["total"].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        for k, v in losses.items():
            train_sums[k] = train_sums.get(k, 0.0) + v.item()
        n += 1

    val = evaluate(model, loaders["val"])
    history.append({"epoch": epoch, "lr": optimizer.param_groups[0]["lr"],
                    **{f"train_{k}": v / n for k, v in train_sums.items()},
                    **{f"val_{k}": v for k, v in val.items()}})
    if val["total"] < best_val:
        best_val, best_state = val["total"], copy.deepcopy(model.state_dict())
    scheduler.step()

    print(f"epoch {epoch:3d} | train {train_sums['total'] / n:.4f} | val {val['total']:.4f} | "
          + " ".join(f"{k}={val[k]:.3f}" for k in LOSS_WEIGHTS)
          + f" | {(time.time() - t0) / 60:.1f} min")

model.load_state_dict(best_state)
history = pd.DataFrame(history)
test_metrics = evaluate(model, loaders["test"])
print("test:", {k: round(v, 4) for k, v in test_metrics.items()})

torch.save({
    "model_state_dict": model.state_dict(),
    "ae_small_cols": AE_SMALL_COLS,
    "ae_context_cols": AE_CONTEXT_COLS,
    "text_cols": TEXT_COLS,
    "mobility_cols": MOB_COLS,
    "mobility_scalar_mean": mob_mean,
    "mobility_scalar_std": mob_std,
    "history": history.to_dict("records"),
    "test_metrics": test_metrics,
    "config": {"AE_PREPROCESSING": "row_l2_normalize", "TEMPERATURE": TEMPERATURE,
               "LOSS_WEIGHTS": LOSS_WEIGHTS, "LR": LR, "WEIGHT_DECAY": WEIGHT_DECAY,
               "NUM_EPOCHS": NUM_EPOCHS, "SEED": SEED},
}, CKPT_PATH)
history.to_csv(CONTRASTIVE_DIR / "train_history.csv", index=False)
print("saved:", CKPT_PATH)

## 4. Training diagnostics and cross-modal retrieval

Plot the total and pairwise validation losses. On up to 1,024 test POIs, compute Recall@K for paired cross-modal retrieval. The random baseline is K divided by the number of sampled POIs.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["epoch"], history["train_total"], label="train")
axes[0].plot(history["epoch"], history["val_total"], label="val")
axes[0].set(title="Total loss", xlabel="epoch")
axes[0].legend()
for k in LOSS_WEIGHTS:
    axes[1].plot(history["epoch"], history[f"val_{k}"], label=k)
axes[1].set(title="Validation loss by pair", xlabel="epoch")
axes[1].legend()
plt.tight_layout()
plt.show()


@torch.no_grad()
def collect_embeddings(model, loader, max_samples=1024):
    model.eval()
    out, n = {}, 0
    for batch in loader:
        z = model(**to_device(batch))
        take = min(len(batch["ae_small"]), max_samples - n)
        for k, v in z.items():
            out.setdefault(k, []).append(v[:take].cpu())
        n += take
        if n >= max_samples:
            break
    return {k: torch.cat(v) for k, v in out.items()}


def recall_at_k(query, key, ks=(1, 5, 10)):
    ranks = torch.argsort(query @ key.T, dim=1, descending=True)
    target = torch.arange(len(query))[:, None]
    return {f"R@{k}": (ranks[:, :k] == target).any(dim=1).float().mean().item() for k in ks}


emb = collect_embeddings(model, loaders["test"])
pairs = {"AE-Text": ("ae_small", "text"), "AE-Mobility": ("ae_small", "mobility"),
         "Text-Mobility": ("text", "mobility"), "AE-small-context": ("ae_small", "ae_context")}
pd.DataFrame({name: recall_at_k(emb[a], emb[b]) for name, (a, b) in pairs.items()}).T.round(3)

## 5. Project the regional AlphaEarth field to BEACON embeddings

Read each 64-band GeoTIFF in blocks, L2-normalize every valid pixel, apply the frozen `AEProjector`, and write a 128-band GeoTIFF on the same grid.


In [ ]:
import rasterio

PROJECT_BATCH = 65536
BEACON_DIR.mkdir(parents=True, exist_ok=True)

ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
model = BEACON()
model.load_state_dict(ckpt["model_state_dict"])
projector = model.ae_projector.to(DEVICE).eval()


@torch.no_grad()
def project_block(block):
    """[64, h, w] → [128, h, w]"""
    c, h, w = block.shape
    x = l2_normalize_np(np.nan_to_num(block, nan=0.0, posinf=0.0, neginf=0.0).reshape(c, -1).T)
    z = np.concatenate([
        projector(torch.from_numpy(x[i:i + PROJECT_BATCH]).to(DEVICE)).cpu().numpy()
        for i in range(0, len(x), PROJECT_BATCH)
    ])
    return z.T.reshape(-1, h, w).astype(np.float32)


tif_paths = sorted(AE64_DIR.glob("*.tif"))
print("tiles:", len(tif_paths))

for src_path in tif_paths:
    out_path = BEACON_DIR / f"{src_path.stem}_z128.tif"
    if out_path.exists():
        continue
    with rasterio.open(src_path) as src:
        assert src.count == 64, f"{src_path.name}: expected 64 bands, got {src.count}"
        profile = src.profile.copy()
        profile.update(count=128, dtype="float32", compress="LZW", BIGTIFF="YES")
        with rasterio.open(out_path, "w", **profile) as dst:
            for _, window in src.block_windows(1):
                dst.write(project_block(src.read(window=window, out_dtype="float32")), window=window)
    print("saved:", out_path.name)

In [ ]:
# Verify the band count, coordinate reference system, and grid against each input
rows = []
for src_path in tif_paths:
    out_path = BEACON_DIR / f"{src_path.stem}_z128.tif"
    with rasterio.open(src_path) as a, rasterio.open(out_path) as b:
        rows.append({"tile": src_path.name, "bands": b.count,
                     "same_grid": a.crs == b.crs and a.transform == b.transform and a.shape == b.shape})
check = pd.DataFrame(rows)
print(check["bands"].value_counts().to_dict(), "| all same grid:", check["same_grid"].all())